# RLFT: constrained generation with a GRADED reward

This notebook is a **thin demo**: every step calls into the tested `geap_tuning`
package. It mirrors [`examples/run_rlft_constrained.py`](../examples/run_rlft_constrained.py).

Our earlier RLFT work on verifiable math produced **two documented null
results** (see [`docs/doe/rlft-reward-ranking/README.md`](../docs/doe/rlft-reward-ranking/README.md)):
the base saturated the task, and a binary reward could not teach a format the
base never emitted. This demo answers **both**. Each prompt spells out four kinds
of constraint at once (required keywords, forbidden filler words, a word-count
band, a sentence-count band). The reward is the **fraction** of independently
checked components satisfied — so it has variance even for a mediocre rollout
(gradient), and satisfying four constraint types at once leaves headroom even for
a strong base.

A **pilot gate** scores the untuned base first and refuses to spend unless the
base has headroom (constraint `accuracy` below `SAT_CEILING`).

> **Requires live GCP and incurs tuning cost** (one RLFT job). `gemini-3.5-flash`
> is the only RLFT-supported base; the tuning client stays **regional** (the
> `global` endpoint serves Gemini 3.x *inference* but not tuning), while the
> untuned baseline runs against that global inference endpoint.

In [ ]:
from geap_tuning.config import genai_client, load_config

BASE_MODEL = "gemini-3.5-flash"  # only RLFT-supported base
SAT_CEILING = 0.85  # base constraint accuracy must be below this to have headroom
cfg = load_config()
client = genai_client(cfg)  # tuning is regional-only; global excludes tuning
cfg

## 1. Build the constrained-generation dataset

Every `references` value is a string (the RLFT record format requires it), and
every band is jointly satisfiable by construction. Records carry a user turn, the
`references`, and a **neutral** system instruction — no answer marker handed out
for free.

In [ ]:
from geap_tuning.rlft.constrained import (
    CONSTRAINT_SPECS,
    build_constrained_dataset,
    build_records,
    split_dataset,
)

paths = build_constrained_dataset("../datasets/rlft_constrained")
train_specs, _, test_specs = split_dataset(CONSTRAINT_SPECS)
train_records = build_records(train_specs)
test_records = build_records(test_specs)
print(f"{len(train_records)} train, {len(test_records)} test prompts")
print(train_specs[0].prompt)

## 2. The graded reward

`constraint_reward.evaluate` is a real, unit-tested, stdlib-only function shipped
**verbatim** to the GEAP code-execution sandbox (via `build_reward_config`) and
reused for offline eval. It returns the fraction of components satisfied, in
`[0, 1]` — partial credit is what gives the training signal variance.

In [ ]:
from geap_tuning.rlft import constraint_reward

refs = train_specs[0].references
full = {"parts": [{"text": train_specs[0].prompt}]}  # obviously imperfect; illustrative
constraint_reward.component_breakdown(refs, full), constraint_reward.evaluate({"references": refs}, full)

## 3. Preflight the reward

`validate_reward_config` scores the reward on one example before we spend money —
RLFT auto-stops if >80% of reward calls error. `validate_reward` is regional, so
it runs in `cfg.location`.

In [ ]:
from geap_tuning.rlft.tune import build_reward_config, validate_reward_config

reward_cfg = build_reward_config("constraint_satisfaction", module=constraint_reward)
preflight = validate_reward_config(
    client,
    project=cfg.project,
    location=cfg.location,
    sample_answer="A short reply that mentions the required keywords.",
    example_record=train_records[0],
    reward_config=reward_cfg,
)
preflight

## 4. Pilot gate — score the untuned base before spending

Gemini 3.x *inference* runs on the `global` endpoint, so the baseline client
routes there. Proceed only if the base has headroom (accuracy below the ceiling).

In [ ]:
from geap_tuning.inference import generate
from geap_tuning.rlft.constraint_eval import run_eval

base_client = genai_client(cfg, base_model=BASE_MODEL)
baseline = run_eval(
    test_records,
    generate_fn=lambda u, s: generate(base_client, BASE_MODEL, u, system_instruction=s),
)
has_headroom = baseline["accuracy"] < SAT_CEILING
print(
    f"untuned {BASE_MODEL}: accuracy={baseline['accuracy']:.3f} "
    f"full_satisfaction_rate={baseline['full_satisfaction_rate']:.3f} (ceiling {SAT_CEILING})"
)
print("Pilot gate:", "PASSED — headroom confirmed" if has_headroom else "FAILED — base already saturated")

## 5. Stage to GCS, launch the RLFT job, and wait

In [ ]:
from geap_tuning.gcs import upload_file
from geap_tuning.jobs import find_tuning_job_by_display_name, tuned_endpoint, wait_for_tuning_job
from geap_tuning.rlft.tune import launch_rlft_job

train_uri = upload_file(paths["train"], f"{cfg.bucket}/rlft_constrained/train.jsonl")
val_uri = upload_file(paths["val"], f"{cfg.bucket}/rlft_constrained/val.jsonl")

DISPLAY_NAME = "geap-rlft-constrained"
job = find_tuning_job_by_display_name(client, DISPLAY_NAME)
if job is None:
    job = launch_rlft_job(
        client,
        train_uri=train_uri,
        val_uri=val_uri,
        display_name=DISPLAY_NAME,
        base_model=BASE_MODEL,
        reward_config=reward_cfg,
        labels=cfg.labels,
    )
job = wait_for_tuning_job(client, job.name)
endpoint = tuned_endpoint(job)
endpoint

## 6. Score the tuned endpoint and report the lift

The tuned endpoint lands on its own (multi-)region, so route eval there. Report
the headline accuracy lift plus a bootstrap 95% CI on the full-satisfaction rate.

In [ ]:
from geap_tuning.config import genai_client_for_endpoint
from geap_tuning.rlft.evaluate import bootstrap_ci

eval_client = genai_client_for_endpoint(cfg, endpoint)
tuned = run_eval(
    test_records,
    generate_fn=lambda u, s: generate(eval_client, endpoint, u, system_instruction=s),
)
print(
    f"TUNED accuracy={tuned['accuracy']:.3f} full_satisfaction_rate={tuned['full_satisfaction_rate']:.3f}"
)
print(f"LIFT accuracy {baseline['accuracy']:.3f}->{tuned['accuracy']:.3f} (+{tuned['accuracy'] - baseline['accuracy']:.3f})")

b_low, b_high = bootstrap_ci(int(baseline["full_satisfaction_hits"]), int(baseline["n"]))
t_low, t_high = bootstrap_ci(int(tuned["full_satisfaction_hits"]), int(tuned["n"]))
print(f"full_satisfaction base CI[{b_low:.3f}, {b_high:.3f}] -> tuned CI[{t_low:.3f}, {t_high:.3f}]")